# Contextual Compression

Shrink each retrieved chunk to only the sentences that matter for the question.

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
def compress(query, chunk):
    return llm(
        "Extract only the sentences from the passage that help answer the question. "
        "Copy them verbatim. If none are relevant reply with NONE.\n\n"
        f"Question: {query}\n\nPassage: {chunk}"
    )

query = "How is mean reciprocal rank calculated?"
hits = store.search(query, 5)
compressed = [c for c in (compress(query, h["text"]) for h in hits) if c.upper() != "NONE"]

before = sum(len(h["text"]) for h in hits)
after = sum(len(c) for c in compressed)
print(f"context shrank from {before} to {after} characters")

In [ ]:
print(answer(query, compressed))

Compression saves tokens and removes distractors, at the cost of one extra LLM call per chunk. Use a small, fast model for it.